# Order Permutation Test for Cognitive Foundation Models

This notebook walks through the **order permutation test** from `order_permutation_test.py`. We test whether language models fine-tuned on psychological data produce predictions that are **invariant to the ordering of exchangeable context trials**.

Under exchangeability, shuffling the order of context trials should not change the predictive distribution for a held-out target trial.

## What is order exchangeability?

Imagine you're watching a model predict human choices in a psychology experiment. You show it 10 previous trials, then ask: "What will the human do next?"

If the trials are **exchangeable** (no learning, no feedback, no adaptive procedure), shuffling the order of the 10 context trials should give the same prediction. The *information content* is what matters, not the *ordering*.

**The test:** For each participant, we fix a target trial and generate M random permutations of the context. We measure the variance of the predicted probabilities across permutations. Under perfect exchangeability, variance = 0.

## Step 1: Load the data

We use the [Psych-101](https://huggingface.co/datasets/marcelbinz/Psych-101) test set. Each entry is one participant's full experimental session as a text prompt.

In [1]:
from datasets import load_dataset

ds = load_dataset("marcelbinz/Psych-101-test", split="test")
print(f"Total entries: {len(ds)}")
print(f"Columns: {ds.column_names}")

Total entries: 6561
Columns: ['text', 'experiment', 'participant']


We test on two experiments chosen to represent contrasting exchangeability regimes:

| Experiment | Task | Exchangeable? | Role |
|---|---|---|---|
| `hebart2023things` | Triplet odd-one-out similarity (3 choices) | **Yes** — random independent triplets from ~1B combinations | **Primary test** |
| `ruggeri2022globalizability` | Intertemporal choice via adaptive staircase (2 choices) | **No** — branching titration, each trial depends on previous response | **Negative control** |

**THINGS odd-one-out** (Hebart et al., 2023): Participants see randomly sampled triplets of objects (from 1,854 concepts) and pick the odd one out. No feedback, no adaptive procedure — trials are i.i.d. by design.

**Intertemporal choice** (Ruggeri et al., 2022): Participants choose between immediate vs. delayed rewards through an adaptive staircase. If they choose immediate, the delayed reward escalates; when they switch, the procedure advances to the next scenario. Each trial is determined by the previous response — trials are *not* exchangeable. This serves as a negative control: we *expect* order sensitivity here, validating that the test can detect sequential dependencies.

In [3]:
hebart = [ex for ex in ds if ex["experiment"].startswith("hebart2023")]
ruggeri = [ex for ex in ds if ex["experiment"].startswith("ruggeri2022")]
print(f"hebart2023things: {len(hebart)} participants")
print(f"ruggeri2022globalizability: {len(ruggeri)} participants")

hebart2023things: 1218 participants
ruggeri2022globalizability: 1295 participants


## Step 2: What does one participant's data look like?

Each participant's data is a single text string: an instruction header followed by sequential trial lines. Let's look at one.

In [5]:
# Print the first 500 characters of one participant
sample = hebart[0]["text"]
print(sample[:777])
print(f"\n... ({len(sample)} characters total)")

You will be presented with triplets of objects, which will be assigned to the keys T, Z, and R.
In each trial, please indicate which object you think is the odd one out by pressing the corresponding key.
In other words, please choose the object that is the least similar to the other two.

T: bottle opener, Z: ironing board, and R: goalpost. You press <<R>>.
T: calculator, Z: wrist, and R: laptop. You press <<Z>>.
T: tool, Z: chain, and R: quad. You press <<R>>.
T: birdbath, Z: bison, and R: jellyfish. You press <<T>>.
T: dreidel, Z: record player, and R: swing set. You press <<R>>.
T: green beans, Z: headlamp, and R: coral. You press <<Z>>.
T: waterwheel, Z: peppermint, and R: dandelion. You press <<R>>.
T: snowmobile, Z: brooch, and R: fishing pole. You press <<Z>>.

... (3782 characters total)


Notice the structure:
- **Instruction header**: tells the model what the task is and which keys are used
- **Trial lines**: each line has a stimulus and a response in `<<X>>` markers
- **Response tokens vary per participant** — one might use `{V,K,H}`, another `{S,U,Q}`

We need to parse this into structured data.

## Step 3: Parsing a participant

We split each participant's text into an instruction, a list of trials, and the valid response tokens.

In [6]:
import re
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class Trial:
    stimulus: str   # e.g., "V: train, K: moccasin, and H: wedge."
    response: str   # e.g., "V"
    full_text: str  # e.g., "V: train, K: moccasin, and H: wedge. You press <<V>>."

@dataclass
class Participant:
    instruction: str
    trials: List[Trial]
    response_tokens: List[str]  # e.g., ["V", "K", "H"]
    participant_id: int

In [7]:
def extract_response_tokens_from_instruction(instruction):
    """Extract response token letters from the instruction header."""
    # Three-token: "keys X, Y, and Z"
    m = re.search(r'keys?\s+([A-Z]),\s*([A-Z]),?\s*and\s+([A-Z])', instruction)
    if m:
        return [m.group(1), m.group(2), m.group(3)]
    # Two-token: "options X and Y"
    m = re.search(r'options?,?\s*(?:labeled\s+)?([A-Z])\s+and\s+([A-Z])', instruction)
    if m:
        return [m.group(1), m.group(2)]
    return None

def extract_response_tokens_fallback(text):
    """Fallback: scan for unique <<X>> tokens."""
    return sorted(set(m.group(1) for m in re.finditer(r"<<([^>]+)>>", text)))

In [8]:
def parse_participant(text, participant_id):
    """Parse a Psych-101 text entry into a Participant."""
    lines = text.strip().split('\n')

    trial_lines, instruction_lines = [], []
    found_first_trial = False

    for line in lines:
        stripped = line.strip()
        if not stripped:
            if not found_first_trial:
                instruction_lines.append(stripped)
            continue
        if '<<' in stripped and '>>' in stripped:
            found_first_trial = True
            trial_lines.append(stripped)
        elif not found_first_trial:
            instruction_lines.append(stripped)

    instruction = '\n'.join(instruction_lines).strip()

    response_tokens = extract_response_tokens_from_instruction(instruction)
    if response_tokens is None:
        response_tokens = extract_response_tokens_fallback(text)

    trials = []
    for line in trial_lines:
        resp_match = re.search(r'<<([^>]+)>>', line)
        if not resp_match:
            continue
        response = resp_match.group(1)
        stim_match = re.search(r'^(.+?)\s*You press\s*<<', line)
        stimulus = stim_match.group(1).strip() if stim_match else line[:line.index('<<')].strip()
        trials.append(Trial(stimulus=stimulus, response=response, full_text=line.strip()))

    return Participant(instruction=instruction, trials=trials,
                       response_tokens=response_tokens, participant_id=participant_id)

In [9]:
# Parse one participant and inspect the result
p = parse_participant(hebart[0]["text"], participant_id=0)

print(f"Response tokens: {p.response_tokens}")
print(f"Number of trials: {len(p.trials)}")
print(f"\nInstruction (first 200 chars):\n{p.instruction[:200]}")
print(f"\nFirst 3 trials:")
for t in p.trials[:3]:
    print(f"  Stimulus: {t.stimulus}")
    print(f"  Response: {t.response}")
    print()

Response tokens: ['T', 'Z', 'R']
Number of trials: 60

Instruction (first 200 chars):
You will be presented with triplets of objects, which will be assigned to the keys T, Z, and R.
In each trial, please indicate which object you think is the odd one out by pressing the corresponding k

First 3 trials:
  Stimulus: T: bottle opener, Z: ironing board, and R: goalpost.
  Response: R

  Stimulus: T: calculator, Z: wrist, and R: laptop.
  Response: Z

  Stimulus: T: tool, Z: chain, and R: quad.
  Response: R



## Step 4: Building prompts

To get the model's prediction, we build a prompt from:
1. The instruction header
2. Some context trials (past behavior)
3. A target stimulus ending with `"You press <<"` — the model completes this

In [10]:
def build_prompt(instruction, context_trials, target_stimulus):
    """Build a prompt ending with 'You press <<' for the model to complete."""
    parts = [instruction, ""]
    for trial in context_trials:
        parts.append(trial.full_text)
    parts.append(f"{target_stimulus} You press <<")
    return '\n'.join(parts)

In [11]:
# Example: use the first 5 trials as context, predict the 6th
context = p.trials[:5]
target = p.trials[5]

prompt = build_prompt(p.instruction, context, target.stimulus)
print("=== PROMPT (last 300 chars) ===")
print(f"...{prompt[-300:]}")
print(f"\nThe model sees this and predicts one of: {p.response_tokens}")
print(f"The human actually pressed: {target.response}")

=== PROMPT (last 300 chars) ===
...ou press <<R>>.
T: calculator, Z: wrist, and R: laptop. You press <<Z>>.
T: tool, Z: chain, and R: quad. You press <<R>>.
T: birdbath, Z: bison, and R: jellyfish. You press <<T>>.
T: dreidel, Z: record player, and R: swing set. You press <<R>>.
T: green beans, Z: headlamp, and R: coral. You press <<

The model sees this and predicts one of: ['T', 'Z', 'R']
The human actually pressed: Z


## Step 5: Getting the predictive distribution

We run the prompt through the model, take logits at the last token position (right after `<<`), restrict to only valid response tokens, and softmax.

```
Prompt: "... You press <<"
                          ^
                     logits here → restrict to {V, K, H} → softmax → {V: 0.72, K: 0.15, H: 0.13}
```

In [12]:
import torch

def get_predictive_distribution(model, tokenizer, prompt, token_id_map, device):
    """Run one forward pass and return P(response_token) for each valid token."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(input_ids)

    # Logits at the last position (the token the model predicts next)
    logits = outputs.logits[0, -1, :]

    # Restrict to valid response tokens and softmax
    token_ids = list(token_id_map.values())
    token_names = list(token_id_map.keys())
    probs = torch.softmax(logits[token_ids].float(), dim=0).cpu().numpy()

    return {name: float(prob) for name, prob in zip(token_names, probs)}

## Step 6: The Order Permutation Test

**Question:** Does shuffling the context trials change the model's prediction?

**Method:**
1. Take a participant with trials `[A, B, C, D, E, F, ...]`
2. Pick a target trial (e.g., `F`)
3. The context is `[A, B, C, D, E]`
4. Create M random permutations of the context
5. For each permutation, get the model's prediction for `F`
6. If the model is exchangeable, all M predictions should be identical

```
Original:       instruction + [A, B, C, D, E] + "F stimulus. You press <<" → {V: 0.72, K: 0.15, H: 0.13}
Permutation 1:  instruction + [C, A, D, B, E] + "F stimulus. You press <<" → {V: 0.70, K: 0.17, H: 0.13}
Permutation 2:  instruction + [E, B, A, D, C] + "F stimulus. You press <<" → {V: 0.68, K: 0.18, H: 0.14}
```

The instruction and the target trial are **always the same**. Only the order of context trials changes. Each permutation contains exactly the same trials — same information content, different ordering.

**Metric:** Variance of each probability across permutations. Should be 0 under exchangeability.

In [13]:
import numpy as np

def test1_one_participant(model, tokenizer, p, token_id_map, device,
                          n_permutations=5, seed=3407, show_prompts=True):
    """Run the order permutation test for a single participant on their last target trial."""
    rng = np.random.default_rng(seed)
    n_trials = len(p.trials)
    target_idx = n_trials - 1                # predict the last trial
    context_trials = p.trials[:target_idx]
    target_trial = p.trials[target_idx]

    print(f"Participant has {n_trials} trials")
    print(f"Context: trials 0..{target_idx-1} ({len(context_trials)} trials)")
    print(f"Target:  trial {target_idx} — stimulus: {target_trial.stimulus}")
    print(f"Human response: {target_trial.response}")
    print(f"Valid tokens: {p.response_tokens}")
    print()

    distributions = []
    for perm_i in range(n_permutations):
        # Shuffle context into a random order
        perm = rng.permutation(len(context_trials))
        shuffled = [context_trials[j] for j in perm]

        prompt = build_prompt(p.instruction, shuffled, target_trial.stimulus)

        if show_prompts:
            print(f"--- Permutation {perm_i} ---")
            print(f"  Context order (trial indices): {list(perm[:6])}{'...' if len(perm) > 6 else ''}")
            # Show last 3 context trial lines + target line to illustrate what changes
            n_show = 3
            print(f"  Prompt tail (last {n_show} context trials + target):")
            for j, trial in enumerate(shuffled[-n_show:]):
                orig_idx = perm[len(shuffled) - n_show + j]
                print(f"    [orig #{orig_idx:>2d}] {trial.full_text[:90]}{'...' if len(trial.full_text) > 90 else ''}")
            print(f"    [TARGET ] {target_trial.stimulus} You press <<___")
            print()

        dist = get_predictive_distribution(model, tokenizer, prompt, token_id_map, device)
        distributions.append(dist)
        print(f"  → P = {dist}")
        print()

    # Compute variance across permutations for each token
    print("=" * 60)
    print("Variance across permutations:")
    for tok in p.response_tokens:
        probs = [d[tok] for d in distributions]
        print(f"  Var(P({tok})) = {np.var(probs, ddof=1):.8f}")
    mean_var = np.mean([np.var([d[tok] for d in distributions], ddof=1) for tok in p.response_tokens])
    print(f"  Mean variance = {mean_var:.8f}")

## Step 7: Load a model and run the test

Now let's load a model and run the order permutation test on one participant. We use `unsloth` for fast inference.

In [ ]:
import unsloth  # noqa: F401 — must be imported before transformers
from unsloth import FastLanguageModel

MODEL_NAME = "socius/Qwentaur-0.6B-LoRA-r16"  # change to any model you want to test (e.g., socius/Llama-Centaur-1B-LoRA-r16)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=32768,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(model)
device = next(model.parameters()).device
print(f"Loaded {MODEL_NAME} on {device}")

In [15]:
# Build the token ID map for this participant
# Maps each response letter to its token ID in the model's vocabulary
token_id_map = {}
for tok in p.response_tokens:
    ids = tokenizer.encode(tok, add_special_tokens=False)
    if len(ids) == 1:
        token_id_map[tok] = ids[0]
    else:
        ids = tokenizer.encode(f" {tok}", add_special_tokens=False)
        token_id_map[tok] = ids[-1]

print(f"Token ID map: {token_id_map}")

Token ID map: {'T': 51, 'Z': 57, 'R': 49}


### Run the order permutation test on one participant

Below, for each permutation we print:
- The **context order** (which trial indices appear in what position)
- The **last 3 context lines + target line** of the prompt — notice how the ending is always the same (same target stimulus), but the preceding trials are shuffled
- The model's **predicted distribution** over response tokens

In [16]:
test1_one_participant(model, tokenizer, p, token_id_map, device, n_permutations=5)

Participant has 60 trials
Context: trials 0..58 (59 trials)
Target:  trial 59 — stimulus: T: pepper, Z: sauerkraut, and R: aardvark.
Human response: Z
Valid tokens: ['T', 'Z', 'R']

--- Permutation 0 ---
  Context order (trial indices): [53, 46, 16, 4, 35, 6]...
  Prompt tail (last 3 context trials + target):
    [orig # 1] T: calculator, Z: wrist, and R: laptop. You press <<Z>>.
    [orig #10] T: flashbulb, Z: dynamite, and R: coffee. You press <<Z>>.
    [orig #54] T: traffic light, Z: headlamp, and R: gumball. You press <<R>>.
    [TARGET ] T: pepper, Z: sauerkraut, and R: aardvark. You press <<___

  → P = {'T': 0.027960943058133125, 'Z': 0.04609980061650276, 'R': 0.9259392619132996}

--- Permutation 1 ---
  Context order (trial indices): [35, 54, 14, 51, 33, 11]...
  Prompt tail (last 3 context trials + target):
    [orig #57] T: puck, Z: headphones, and R: bulletproof vest. You press <<T>>.
    [orig #20] T: goldfish, Z: ski boots, and R: racehorse. You press <<Z>>.
    [orig #32

## Running the full pipeline

The notebook above demonstrates the core logic for a single participant. The full pipeline scales this across all participants and models:

```bash
# 1. Run permutation test (GPU required)
python order_permutation_test.py \
    --model socius/Qwentaur-8B-LoRA-r16 \
    --experiments hebart2023things ruggeri2022globalizability \
    --n-permutations 50 \
    --output-dir results/Qwentaur-8B

# 2. Compute aggregate statistics from raw results
python analyse_permutation.py

# 3. Generate figures (reads CSVs produced by step 2)
python generate_permutation_plots.py
```

**Expected results:**
- On **THINGS odd-one-out** (primary test): fine-tuned models show ~8.4× lower order variance than base models — they learned that trials are exchangeable
- On **Intertemporal choice** (negative control): fine-tuned models show ~2× *higher* order variance — they learned the sequential staircase structure, so shuffling disrupts predictions

## Cleanup

In [ ]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")